# NeuroScan: data exploration

Exploration of the [Brain Tumor MRI dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)
before any modeling. Training, evaluation and model plots live in the `neuroscan` package and in MLflow;
this notebook only looks at the data.

Data is loaded with `neuroscan.data` (the same code training uses). Download it first with `make data`.

**Dataset version.** The current Kaggle version has **7,200 images** (1,400 per class in `Training`,
400 per class in `Testing`). An earlier version had 7,023 images.

In [ ]:
import hashlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

from neuroscan.config import CLASS_NAMES
from neuroscan.data.datasets import scan_raw
from neuroscan.data.preprocess import preprocess_file

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
CACHE = ROOT / "data" / "cache"
CACHE.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(zip(CLASS_NAMES, ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]))

df = scan_raw(RAW)
df.head()

## 1. Class distribution

In [ ]:
counts = df.groupby(["kaggle_split", "label"]).size().unstack()[list(CLASS_NAMES)]
display(counts.assign(total=counts.sum(axis=1)))

ax = counts.T.plot.bar(figsize=(8, 3.5), rot=0, color=["#8172B3", "#937860"])
ax.set_ylabel("images")
ax.set_title("Images per class and Kaggle folder")
for c in ax.containers:
    ax.bar_label(c, fontsize=8)
plt.show()

The classes are perfectly balanced, so accuracy and macro-F1 behave similarly here. The 78/22 Kaggle
Training/Testing ratio won't be used: the project builds its own duplicate-aware 70/15/15 split
(`splits/v1.csv`).

## 2. Sample images

In [ ]:
rng = np.random.default_rng(0)
n = 5
fig, axes = plt.subplots(len(CLASS_NAMES), n, figsize=(2.2 * n, 2.3 * len(CLASS_NAMES)))
for r, name in enumerate(CLASS_NAMES):
    paths = df[df.label == name].path.to_numpy()
    for c, rel in enumerate(rng.choice(paths, n, replace=False)):
        ax = axes[r, c]
        ax.imshow(Image.open(RAW / rel).convert("RGB"))
        ax.set_xticks([]); ax.set_yticks([])
        if c == 0:
            ax.set_ylabel(name, fontsize=11)
fig.suptitle("Random raw images per class (original size and aspect)")
plt.tight_layout()
plt.show()

All three views appear (axial, coronal, sagittal), with varying crops, contrast and borders. There's no
view label in the dataset, so view balance per class can't be checked directly.

What the model actually sees after `neuroscan.data.preprocess` (decode → RGB → bilinear resize to 224×224):

In [ ]:
picks = [df[df.label == name].path.iloc[7] for name in CLASS_NAMES]
fig, axes = plt.subplots(2, 4, figsize=(11, 5.5))
for i, rel in enumerate(picks):
    raw = Image.open(RAW / rel)
    axes[0, i].imshow(raw.convert("RGB"))
    axes[0, i].set_title(f"{CLASS_NAMES[i]}\n{raw.size[0]}×{raw.size[1]} {raw.mode}", fontsize=9)
    axes[1, i].imshow(preprocess_file(RAW / rel).astype("uint8"))
    axes[1, i].set_title("model input 224×224", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Image properties

One pass over all images, cached to `data/cache/image_stats.csv`: resolution, file format, color mode,
file size, whether the 3 channels actually differ, brightness and the fraction of near-black background.

In [ ]:
STATS = CACHE / "image_stats.csv"


def image_stats(rel: str) -> dict:
    path = RAW / rel
    data = path.read_bytes()
    img = Image.open(path)
    arr = np.asarray(img.convert("RGB"), dtype=np.int16)
    gray = arr.mean(axis=2)
    return {
        "path": rel,
        "width": img.width,
        "height": img.height,
        "img_format": img.format,
        "color_mode": img.mode,
        "file_kb": len(data) / 1024,
        "md5": hashlib.md5(data).hexdigest(),  # duplicate check only, not security
        "max_channel_diff": int(np.abs(arr - arr[..., :1]).max()),
        "mean_intensity": gray.mean(),
        "std_intensity": gray.std(),
        "background_frac": (gray < 10).mean(),
    }


if STATS.exists():
    stats = pd.read_csv(STATS)
else:
    stats = pd.DataFrame([image_stats(p) for p in df.path])
    stats.to_csv(STATS, index=False)
stats = df.merge(stats, on="path")
stats["aspect"] = stats.width / stats.height
stats["megapixels"] = stats.width * stats.height / 1e6
stats.describe().T.round(2)

### 3.1 Resolution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name in CLASS_NAMES:
    s = stats[stats.label == name]
    axes[0].scatter(s.width, s.height, s=6, alpha=0.35, label=name, color=COLORS[name])
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel("width (px)"); axes[0].set_ylabel("height (px)")
axes[0].set_title("Image size per class (log scale)")
axes[0].legend(markerscale=3)

top = stats.assign(res=stats.width.astype(str) + "×" + stats.height.astype(str))
share = pd.crosstab(top.res, top.label, normalize="columns")[list(CLASS_NAMES)]
share = share.loc[share.max(axis=1).sort_values(ascending=False).index[:8]]
share.plot.barh(ax=axes[1], color=[COLORS[c] for c in CLASS_NAMES])
axes[1].invert_yaxis()
axes[1].set_xlabel("share of the class")
axes[1].set_title("Most common resolutions")
plt.tight_layout()
plt.show()

stats.groupby("label")[["width", "height", "aspect", "file_kb"]].median().round(2)

### 3.2 Format and color mode

In [ ]:
display(pd.crosstab(stats.label, [stats.img_format, stats.color_mode]))
stats["truly_color"] = stats.max_channel_diff > 10
display(
    stats.groupby("label").truly_color.mean().rename("share with differing RGB channels").to_frame()
)

Most MRIs are grayscale stored as RGB (all channels equal or nearly so). Images with real color differences
are worth a look: they may be screenshots or annotated images. The API rejects strongly colored
uploads with the same check.

In [ ]:
colored = stats[stats.truly_color].sort_values("max_channel_diff", ascending=False).head(8)
if len(colored):
    fig, axes = plt.subplots(1, len(colored), figsize=(2.2 * len(colored), 2.6))
    for ax, (_, row) in zip(np.atleast_1d(axes), colored.iterrows()):
        ax.imshow(Image.open(RAW / row.path).convert("RGB"))
        ax.set_title(f"{row.label}\ndiff={row.max_channel_diff}", fontsize=8)
        ax.axis("off")
    plt.suptitle("Images whose RGB channels differ the most")
    plt.tight_layout()
    plt.show()

### 3.3 Intensity and background

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for col, ax, title in [
    ("mean_intensity", axes[0], "Mean intensity"),
    ("std_intensity", axes[1], "Intensity std (contrast)"),
    ("background_frac", axes[2], "Near-black background fraction"),
]:
    bins = np.linspace(stats[col].min(), stats[col].max(), 50)
    for name in CLASS_NAMES:
        ax.hist(stats.loc[stats.label == name, col], bins=bins, histtype="step", lw=1.6,
                density=True, label=name, color=COLORS[name])
    ax.set_title(title)
axes[0].legend()
plt.tight_layout()
plt.show()

## 4. Mean image per class

Average of all images per class after the model's own preprocessing (224×224, grayscale shown).
Clear structure or framing differences between classes hint at acquisition differences the model could
use as shortcuts.

In [ ]:
MEANS = CACHE / "class_means.npz"
if MEANS.exists():
    means = dict(np.load(MEANS))
else:
    means = {
        name: np.mean([preprocess_file(RAW / p).mean(axis=2) for p in df[df.label == name].path], axis=0)
        for name in CLASS_NAMES
    }
    np.savez_compressed(MEANS, **means)

overall = np.mean(list(means.values()), axis=0)
fig, axes = plt.subplots(2, 4, figsize=(13, 6.8))
for i, name in enumerate(CLASS_NAMES):
    axes[0, i].imshow(means[name], cmap="gray", vmin=0, vmax=255)
    axes[0, i].set_title(f"mean: {name}")
    diff = means[name] - overall
    lim = np.abs(diff).max()
    axes[1, i].imshow(diff, cmap="RdBu_r", vmin=-lim, vmax=lim)
    axes[1, i].set_title("difference from overall mean", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Shortcut probe: can metadata alone predict the class?

A random forest sees **only** file and acquisition properties — no pixels of anatomy: resolution, file
size, format, brightness, contrast, background fraction. Chance level is 25%. Anything far above that
means the label is partly recoverable from *how* an image was produced (its source), which a CNN can
exploit too. This is a hint, not a proof; formal source-shortcut tests are out of scope and listed as a
limitation.

In [ ]:
features = stats[["width", "height", "aspect", "file_kb", "mean_intensity", "std_intensity",
                  "background_frac", "max_channel_diff"]].assign(
    is_png=(stats.img_format == "PNG").astype(int)
)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
forest = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1)
scores = cross_val_score(forest, features, stats.label, cv=cv, scoring="accuracy")
print(f"Metadata-only accuracy: {scores.mean():.1%} ± {scores.std():.1%}  (chance: 25%)")

forest.fit(features, stats.label)
imp = pd.Series(forest.feature_importances_, features.columns).sort_values()
imp.plot.barh(figsize=(6, 3.5), title="What the metadata probe relies on")
plt.show()

from sklearn.model_selection import cross_val_predict  # noqa: E402

pred = cross_val_predict(forest, features, stats.label, cv=cv)
pd.crosstab(stats.label, pred, rownames=["true"], colnames=["predicted from metadata"])

## 6. Byte-identical duplicates

Exact file duplicates via MD5. The duplicate audit (`neuroscan.data.dedup`) also uses perceptual hashes to catch resized or
re-encoded near-duplicates, which this check misses.

In [ ]:
groups = stats.groupby("md5")
dup = stats[stats.md5.map(groups.size()) > 1]
print(f"{len(dup)} files in {dup.md5.nunique()} groups of byte-identical images")

per_group = dup.groupby("md5").agg(
    splits=("kaggle_split", lambda s: "+".join(sorted(set(s)))),
    labels=("label", lambda s: "+".join(sorted(set(s)))),
    size=("path", "size"),
)
display(per_group.splits.value_counts().rename("groups").to_frame())
conflicting = per_group[per_group.labels.str.contains("\\+")]
print(f"Groups with conflicting labels: {len(conflicting)}")
display(conflicting.head(10))

In [ ]:
cross = per_group[per_group.splits == "Testing+Training"].head(4)
if len(cross):
    fig, axes = plt.subplots(len(cross), 2, figsize=(5, 2.5 * len(cross)), squeeze=False)
    for r, md5 in enumerate(cross.index):
        rows = stats[stats.md5 == md5].sort_values("kaggle_split")
        for c, (_, row) in enumerate(rows.head(2).iterrows()):
            axes[r, c].imshow(Image.open(RAW / row.path).convert("RGB"))
            axes[r, c].set_title(row.path, fontsize=7)
            axes[r, c].axis("off")
    plt.suptitle("Same file in Training and Testing")
    plt.tight_layout()
    plt.show()

## Findings

Numbers from the run on the 7,200-image version (re-run the notebook to refresh them).

1. **Balanced classes:** 1,800 images per class (1,400 Training + 400 Testing).
2. **"No tumor" comes from a different source.** 86–96% of glioma, meningioma and pituitary images are
   512×512 JPEGs. "No tumor" images are mostly small (median 228×243), with many PNGs, a wide spread of
   resolutions, brighter and higher-contrast pixels, and often no black background. It's also the only
   class with real color images (3.7%).
3. **Metadata predicts the class at ~79% accuracy** (chance: 25%) without seeing any anatomy, and
   **no-tumor is recognized at ~99% recall** from metadata alone. A very high no-tumor recall is therefore
   weak evidence: a CNN can tell no-tumor apart by resolution and contrast,
   not by the absence of a tumor. This is the source-shortcut risk named in the README limitations.
4. **The mean images differ by class:** framing, head size and brightness vary systematically
   (e.g. glioma is darker and framed tighter), another acquisition signal a model can learn.
5. **340 byte-identical files in 153 groups**, all within one Kaggle folder (none across
   Training/Testing and no conflicting labels). Near-duplicates (resized or re-encoded) aren't caught by
   MD5; the duplicate audit finds them with perceptual hashes and builds a split where no duplicate group crosses
   train/val/test.

**Consequences:** treat the tumor-vs-no-tumor sensitivity/specificity as
optimistic; report per-class metrics, not just the average; mention the metadata probe in the
limitations and the model card.